In [15]:
from bokeh.plotting import figure, show, output_file
from bokeh.models import Label
from bokeh.io import export_png, curdoc
curdoc().theme = 'contrast'

import numpy as np
import pandas as pd
from scipy.optimize import fsolve

In [16]:
data = pd.read_csv('methanol_water_vle_data.csv')
x_methanol = data['x_methanol']
y_methanol = data['y_methanol']

# Create a new plot with a title and axis labels
p = figure(title='McCabe-Thiele Diagram for Methanol-Water System', 
           x_axis_label='x_MeOH', y_axis_label='y_MeOH',
           width=800, height=600)

# Constants for operating lines
R, x_D = 1.25, 0.90  # Reflux ratio and distillate composition
B, x_B = 2.0, 0.05   # Boilup ratio and bottoms composition
z = 0.55             # Feed composition

# Define operating line functions
def top_line(x):
    return R / (R + 1) * x + x_D / (R + 1)

def bottom_line(x):
    return (B + 1) / B * x - x_B / B

# Finding intersection of operating lines
intersection_x = fsolve(lambda x: top_line(x) - bottom_line(x), 0.5)[0]
intersection_y = top_line(intersection_x)

# Define a function to plot line and markers
def plot_line_and_markers(x_range, function, color, legend_label, line_dash='dashed'):
    x_vals = np.linspace(*x_range, 100)
    y_vals = function(x_vals)
    p.line(x_vals, y_vals, color=color, legend_label=legend_label, line_dash=line_dash)

# Plot VLE data and diagonal line
p.circle(x_methanol, y_methanol, size=4, color='blue')
p.line(x_methanol, y_methanol, color='blue')
p.line([0, 1], [0, 1], color='black')

# Plot operating lines
plot_line_and_markers((intersection_x, x_D), top_line, 'dodgerblue', 'Rectifying')
plot_line_and_markers((x_B, intersection_x), bottom_line, 'dodgerblue', 'Stripping', 'dotdash')

# q-line
p.line([z, intersection_x], [z, intersection_y], color='black', line_dash='dashed')

# Add points for distillate, bottoms, and feed compositions
for point, label in zip([x_D, x_B, z], [r'$x_D$', r'$x_B$', r'$z$']):
    p.line([point, point], [0, point], color='black', line_dash='dashed')

# Add McCabe-Thiele stage-stepping
stage_x, stage_y = [x_D], [x_D]
while stage_x[-1] > x_B:
    new_x = np.interp(stage_y[-1], y_methanol, x_methanol)
    new_y = top_line(new_x) if new_x > intersection_x else bottom_line(new_x)
    if new_x < x_B:
        new_y = new_x

    stage_x.extend([new_x, new_x])
    stage_y.extend([stage_y[-1], new_y])

    p.line(stage_x[-3:], stage_y[-3:], color='black')

# Add labels and legends
p.legend.title = 'Operating Lines'
p.legend.location = 'top_left'

# Output to static HTML file
output_file('mccabe_thiele_diagram.html')

# Save the plot and open the resulting HTML
show(p)